In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import io
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import torchvision.transforms as transforms
from PIL import Image

# =====================================================================
# GLOBAL CONFIGURATIONS & SEEDING FOR REPRODUCIBILITY
# =====================================================================
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_image_files(directory):
    """Helper to list all images in a directory efficiently."""
    directory = directory.strip()
    if not os.path.exists(directory):
        print(f"⚠️ Warning: Directory does not exist -> '{directory}'")
        return []
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(directory, ext)))
        files.extend(glob.glob(os.path.join(directory, ext.upper())))
    return sorted(files)

# Master storage registry
metadata_records = []

print("=== STARTING DATASET SCAN AND SUBSAMPLING ===")

# =====================================================================
# 1. DF2023 Train Subset (Strict Limit: 10,000 items)
# =====================================================================
df2023_train_img_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15"
df2023_train_gt_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15_GT"

df2023_train_imgs = get_image_files(df2023_train_img_dir)
print(f"Found {len(df2023_train_imgs)} files in DF2023 Train")

if df2023_train_imgs:
    train_sample_size = min(10000, len(df2023_train_imgs))
    sampled_train_imgs = np.random.choice(df2023_train_imgs, size=train_sample_size, replace=False)
    
    for img_path in sampled_train_imgs:
        base_name = os.path.basename(img_path)
        gt_path = os.path.join(df2023_train_gt_dir, base_name)
        metadata_records.append({
            'image_path': img_path,
            'mask_path': gt_path if os.path.exists(gt_path) else None,
            'source_dataset': 'DF2023_Train',
            'category': 'manipulation',
            'split': 'train'
        })

# =====================================================================
# 2. DF2023 Validation & Internal Test Subset (Strict Limit: 3,000 items)
# =====================================================================
df2023_val_img_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15"
df2023_val_gt_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15_GT"

df2023_val_imgs = get_image_files(df2023_val_img_dir)
print(f"Found {len(df2023_val_imgs)} files in DF2023 Val")

if df2023_val_imgs:
    val_sample_size = min(3000, len(df2023_val_imgs))
    sampled_val_imgs = np.random.choice(df2023_val_imgs, size=val_sample_size, replace=False)
    
    df2023_v_idx, df2023_t_idx = train_test_split(sampled_val_imgs, test_size=0.50, random_state=42)
    
    for img_list, split_label in [(df2023_v_idx, 'val'), (df2023_t_idx, 'test')]:
        for img_path in img_list:
            base_name = os.path.basename(img_path)
            gt_path = os.path.join(df2023_val_gt_dir, base_name) 
            metadata_records.append({
                'image_path': img_path,
                'mask_path': gt_path if os.path.exists(gt_path) else None,
                'source_dataset': 'DF2023_Val',
                'category': 'manipulation',
                'split': split_label
            })

# =====================================================================
# 3. SIDA Forensics Subset (Limit to 6,000 items total -> 70/15/15 Split)
# =====================================================================
sida_paths = {
    'full_synthetic': "/kaggle/input/datasets/mubarekahmed/sida-forensics/full_synthetic",
    'masks': "/kaggle/input/datasets/mubarekahmed/sida-forensics/masks",
    'real': "/kaggle/input/datasets/mubarekahmed/sida-forensics/real",
    'tampered': "/kaggle/input/datasets/mubarekahmed/sida-forensics/tampered"
}

sida_all_records = []
for category, path in sida_paths.items():
    sida_imgs = get_image_files(path)
    print(f"Found {len(sida_imgs)} files in SIDA category: {category}")
    for img_path in sida_imgs:
        sida_all_records.append({
            'image_path': img_path,
            'mask_path': None,
            'source_dataset': 'sida-forensics',
            'category': category
        })

sida_df_pool = pd.DataFrame(sida_all_records)
if not sida_df_pool.empty:
    _, sampled_sida_df = train_test_split(
        sida_df_pool, 
        test_size=min(6000, len(sida_df_pool)), 
        stratify=sida_df_pool['category'], 
        random_state=42
    )
    
    sida_train, sida_temp = train_test_split(sampled_sida_df, test_size=0.30, stratify=sampled_sida_df['category'], random_state=42)
    sida_val, sida_test = train_test_split(sida_temp, test_size=0.50, stratify=sida_temp['category'], random_state=42)
    
    sida_train['split'] = 'train'
    sida_val['split'] = 'val'
    sida_test['split'] = 'test'
    
    metadata_records.extend(pd.concat([sida_train, sida_val, sida_test]).to_dict(orient='records'))

# =====================================================================
# 4. So-Fake Setted Combined Subset (Read Parquet Files safely)
# =====================================================================
so_fake_paths = {
    'train': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/train",
    'validation': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/validation",
    'ood_test': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/ood_test"
}

for split_name, path in so_fake_paths.items():
    path = path.strip()
    parquet_files = sorted(glob.glob(os.path.join(path, "*.parquet")))
    print(f"Found {len(parquet_files)} parquet shards in So-Fake: {split_name}")
    
    if parquet_files:
        split_records = []
        for p_file in parquet_files:
            if len(split_records) >= 2000:
                break
            
            df_parquet = pd.read_parquet(p_file)
            
            for _, row in df_parquet.iterrows():
                if len(split_records) >= 2000:
                    break
                
                img_p = row.get('image_path', row.get('path', None))
                mask_p = row.get('mask_path', row.get('mask', None))
                cat = row.get('category', 'synthetic_hybrid')
                
                # Check if the extracted parameters are embedded objects/dicts
                # If they are dictionaries, preserve them intact without casting to strings
                processed_img = img_p if isinstance(img_p, dict) or (isinstance(img_p, str) and img_p) else p_file
                processed_mask = mask_p if isinstance(mask_p, dict) or (isinstance(mask_p, str) and mask_p) else None
                
                split_records.append({
                    'image_path': processed_img,
                    'mask_path': processed_mask,
                    'source_dataset': 'so-fake-combined',
                    'category': cat,
                    'split': 'val' if split_name == 'validation' else ('test' if split_name == 'ood_test' else 'train')
                })
        
        metadata_records.extend(split_records)

# Compile and Verify Final Architecture
columns = ['image_path', 'mask_path', 'source_dataset', 'category', 'split']
final_metadata_df = pd.DataFrame(metadata_records, columns=columns)
final_metadata_df.to_csv("balanced_dataset_metadata.csv", index=False)

print("\n=== FINAL GENERATED METADATA MATRIX ===")
if not final_metadata_df.empty:
    print(final_metadata_df.groupby(['source_dataset', 'split']).size())
else:
    print("❌ Critical: Dataset processing ended with 0 total indexed items.")

# =====================================================================
# 5. PYTORCH MULTI-MODAL MULTI-TASK FORENSIC DATASET
# =====================================================================
class FakeShieldDataset(Dataset):
    def __init__(self, metadata_df, split='train', img_size=(224, 224), metadata_dim=16):
        self.df = metadata_df[metadata_df['split'] == split].reset_index(drop=True)
        self.split = split
        self.img_size = img_size
        self.metadata_dim = metadata_dim
        
        self.transform = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.mask_transform = transforms.Compose([
            transforms.Resize(self.img_size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.df)

    def _load_image_or_mask(self, target, convert_mode='RGB'):
        """Safely parses paths, dictionary records, or binary blobs into PIL Images."""
        if target is None:
            return None
        try:
            if isinstance(target, dict):
                if 'bytes' in target and target['bytes'] is not None:
                    return Image.open(io.BytesIO(target['bytes'])).convert(convert_mode)
                elif 'path' in target and isinstance(target['path'], str) and os.path.exists(target['path']):
                    return Image.open(target['path']).convert(convert_mode)
            elif isinstance(target, str) and os.path.exists(target):
                if not target.endswith('.parquet'):
                    return Image.open(target).convert(convert_mode)
            return None
        except Exception:
            return None

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_entry = row['image_path']
        mask_entry = row['mask_path']
        category = row['category']
        
        # Load Visual Image Stream
        image = self._load_image_or_mask(img_entry, convert_mode='RGB')
        if image is None:
            image = Image.fromarray(np.uint8(np.random.rand(224, 224, 3) * 255))
        image_tensor = self.transform(image)
        
        # Load Ground-Truth Target Masks
        mask = self._load_image_or_mask(mask_entry, convert_mode='L')
        if mask is not None:
            mask_tensor = self.mask_transform(mask)
        else:
            mask_tensor = torch.zeros((1, self.img_size[0], self.img_size[1]))
            
        # Global Binary Classification Mapping
        label = 0 if category == 'real' else 1
        label_tensor = torch.tensor(label, dtype=torch.long)
        
        # Metadata Latent Vectors Formulation
        metadata_vector = torch.randn(self.metadata_dim)
        meta_flag = 1.0 if np.random.rand() > 0.10 else 0.0
        meta_flag_tensor = torch.tensor(meta_flag, dtype=torch.float32)
        
        return image_tensor, metadata_vector, meta_flag_tensor, label_tensor, mask_tensor

# =====================================================================
# 6. MODEL ARCHITECTURE DEFINITION (FAKESHIELD-T PIPELINE)
# =====================================================================
def get_srm_filters():
    filter1 = np.array([[0,  0, 0], [0, -1, 1], [0,  0, 0]], dtype=np.float32)
    filter2 = np.array([[0,  1, 0], [0, -2, 1], [0,  0, 0]], dtype=np.float32)
    filter3 = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float32)
    filters = np.zeros((3, 3, 3, 3), dtype=np.float32)
    for i, f in enumerate([filter1, filter2, filter3]):
        for j in range(3):
            filters[i, j, :, :] = f
    return torch.tensor(filters)

class SRMConvStream(nn.Module):
    def __init__(self):
        super(SRMConvStream, self).__init__()
        self.srm_conv = nn.Conv2d(3, 3, kernel_size=3, padding=1, bias=False)
        self.srm_conv.weight.data.copy_(get_srm_filters())
        self.srm_conv.weight.requires_grad = False 
        
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.SiLU()
        )
    def forward(self, x):
        return self.backbone(self.srm_conv(x))

class TinyViT5MStream(nn.Module):
    def __init__(self, embed_dim=128):
        super(TinyViT5MStream, self).__init__()
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=8, stride=8)
        self.norm = nn.LayerNorm(embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dim_feedforward=512, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
    def forward(self, x):
        patches = self.patch_embed(x)
        B, C, H, W = patches.shape
        patches = self.norm(patches.flatten(2).transpose(1, 2))
        return self.transformer(patches).transpose(1, 2).view(B, C, H, W)

class ModalityAwareFusion(nn.Module):
    def __init__(self, feature_dim=128, metadata_dim=16):
        super(ModalityAwareFusion, self).__init__()
        self.meta_encoder = nn.Sequential(nn.Linear(metadata_dim, feature_dim), nn.ReLU(), nn.Linear(feature_dim, feature_dim))
        self.missing_token = nn.Parameter(torch.randn(1, 1, feature_dim))
        self.cross_attn = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=4, batch_first=True)
        self.norm_visual = nn.LayerNorm(feature_dim)
        self.norm_meta = nn.LayerNorm(feature_dim)
    def forward(self, visual_feats, metadata, meta_present_flags):
        B, C, H, W = visual_feats.shape
        v_flat = visual_feats.flatten(2).transpose(1, 2)
        meta_encoded = self.meta_encoder(metadata).unsqueeze(1)
        missing_expanded = self.missing_token.expand(B, -1, -1)
        flags = meta_present_flags.view(B, 1, 1).float()
        meta_tokens = (flags * meta_encoded) + ((1.0 - flags) * missing_expanded)
        attn_out, _ = self.cross_attn(self.norm_visual(v_flat), self.norm_meta(meta_tokens), meta_tokens)
        return (v_flat + attn_out).transpose(1, 2).view(B, C, H, W)

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DepthwiseSeparableConv, self).__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))

class FPN_PANet_Head(nn.Module):
    def __init__(self, feature_dim=128):
        super(FPN_PANet_Head, self).__init__()
        self.fpn_conv1 = DepthwiseSeparableConv(feature_dim, 64)
        self.fpn_conv2 = DepthwiseSeparableConv(64, 32)
        self.pan_conv1 = DepthwiseSeparableConv(32, 64)
        self.pan_conv2 = DepthwiseSeparableConv(64, 128)
        self.final_mask = nn.Sequential(
            nn.ConvTranspose2d(128, 32, kernel_size=4, stride=4, padding=0), nn.SiLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2, padding=0), nn.Sigmoid()
        )
    def forward(self, x):
        p1 = self.fpn_conv1(x)
        p2 = self.fpn_conv2(F.interpolate(p1, scale_factor=2, mode='nearest'))
        n1 = self.pan_conv1(F.interpolate(p2, scale_factor=0.5, mode='nearest')) + p1
        return self.final_mask(self.pan_conv2(n1))

class FakeShieldT(nn.Module):
    def __init__(self, metadata_dim=16):
        super(FakeShieldT, self).__init__()
        self.artifact_stream = SRMConvStream()
        self.semantic_stream = TinyViT5MStream()
        self.fusion = ModalityAwareFusion(feature_dim=128, metadata_dim=metadata_dim)
        self.localization_head = FPN_PANet_Head(feature_dim=128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.detection_head = nn.Sequential(nn.Linear(128, 32), nn.SiLU(), nn.Linear(32, 2))

    def forward(self, images, metadata, meta_flags):
        combined_feats = self.artifact_stream(images) + self.semantic_stream(images)
        fused_context = self.fusion(combined_feats, metadata, meta_flags)
        pooled = self.global_pool(fused_context).view(fused_context.size(0), -1)
        return self.detection_head(pooled), self.localization_head(fused_context)

# =====================================================================
# 7. MATHEMATICAL EVALUATION ENGINE
# =====================================================================
class FakeShieldEvaluationEngine:
    @staticmethod
    def calculate_miou(pred_masks, gt_masks, threshold=0.5):
        preds = (pred_masks > threshold).float()
        intersection = (preds * gt_masks).sum(dim=(2, 3))
        union = (preds + gt_masks).clamp(0, 1).sum(dim=(2, 3))
        return ((intersection + 1e-7) / (union + 1e-7)).mean().item()

    @staticmethod
    def evaluation_report(all_logits, all_labels, all_pred_masks, all_gt_masks):
        preds_det = torch.argmax(all_logits, dim=1).cpu().numpy()
        labels_det = all_labels.cpu().numpy()
        f1 = f1_score(labels_det, preds_det, average='macro', zero_division=0)
        miou = FakeShieldEvaluationEngine.calculate_miou(all_pred_masks, all_gt_masks)
        return {"F1_Score": f1, "mIoU": miou}

# =====================================================================
# 8. PIPELINE INVOCATION AND METRIC MONITORING
# =====================================================================
if not final_metadata_df.empty:
    print("\n=== INITIALIZING DATA LOADERS WITH MATRIX SLICES ===")
    train_dataset = FakeShieldDataset(final_metadata_df, split='train')
    val_dataset   = FakeShieldDataset(final_metadata_df, split='val')
    test_dataset  = FakeShieldDataset(final_metadata_df, split='test')

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

    print(f"Dataset Split Sizes -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    model = FakeShieldT(metadata_dim=16).to(device)
    criterion_det = nn.CrossEntropyLoss()
    criterion_loc = nn.BCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

    print("\n=== STARTING TRAINING PROCESS ===")
    model.train()
    running_loss = 0.0
    for images, metadata, flags, labels, masks in train_loader:
        images, metadata, flags = images.to(device), metadata.to(device), flags.to(device)
        labels, masks = labels.to(device), masks.to(device)
        
        optimizer.zero_grad()
        logits_det, masks_loc = model(images, metadata, flags)
        
        loss_det = criterion_det(logits_det, labels)
        loss_loc = criterion_loc(masks_loc, masks)
        total_loss = loss_det + 2.0 * loss_loc
        
        total_loss.backward()
        optimizer.step()
        running_loss += total_loss.item()

    print(f"Epoch Complete. Avg Batch Training Loss: {running_loss/len(train_loader):.4f}")

    print("\n=== VALIDATING MODEL PERFORMANCES ===")
    model.eval()
    val_logits, val_labels, val_masks_pred, val_masks_gt = [], [], [], []
    with torch.no_grad():
        for images, metadata, flags, labels, masks in val_loader:
            images, metadata, flags = images.to(device), metadata.to(device), flags.to(device)
            logits_det, masks_loc = model(images, metadata, flags)
            val_logits.append(logits_det.cpu())
            val_labels.append(labels)
            val_masks_pred.append(masks_loc.cpu())
            val_masks_gt.append(masks)

    val_report = FakeShieldEvaluationEngine.evaluation_report(
        torch.cat(val_logits), torch.cat(val_labels), torch.cat(val_masks_pred), torch.cat(val_masks_gt)
    )
    print(f"Validation Performance Metrics -> F1-Score: {val_report['F1_Score']:.4f}, mIoU: {val_report['mIoU']:.4f}")

    print("\n=== STARTING ROBUSTNESS TEST MATRIX ===")
    test_logits, test_labels, test_masks_pred, test_masks_gt = [], [], [], []
    with torch.no_grad():
        for images, metadata, flags, labels, masks in test_loader:
            images, metadata, flags = images.to(device), metadata.to(device), flags.to(device)
            logits_det, masks_loc = model(images, metadata, flags)
            test_logits.append(logits_det.cpu())
            test_labels.append(labels)
            test_masks_pred.append(masks_loc.cpu())
            test_masks_gt.append(masks)

    clean_test_metrics = FakeShieldEvaluationEngine.evaluation_report(
        torch.cat(test_logits), torch.cat(test_labels), torch.cat(test_masks_pred), torch.cat(test_masks_gt)
    )
    print(f"Pristine Base Test F1-Score       : {clean_test_metrics['F1_Score']:.4f}")
    print(f"Pristine Base Test Localization mIoU: {clean_test_metrics['mIoU']:.4f}")
else:
    print("Execution halted due to empty metadata frame extraction.")

=== STARTING DATASET SCAN AND SUBSAMPLING ===
Found 1000000 files in DF2023 Train
Found 5000 files in DF2023 Val
Found 10000 files in SIDA category: full_synthetic
Found 10000 files in SIDA category: masks
Found 10000 files in SIDA category: real
Found 10000 files in SIDA category: tampered
Found 10 parquet shards in So-Fake: train
Found 4 parquet shards in So-Fake: validation
Found 4 parquet shards in So-Fake: ood_test

=== FINAL GENERATED METADATA MATRIX ===
source_dataset    split
DF2023_Train      train    10000
DF2023_Val        test      1500
                  val       1500
sida-forensics    test       900
                  train     4200
                  val        900
so-fake-combined  test      1912
                  train     2000
                  val       2000
dtype: int64

=== INITIALIZING DATA LOADERS WITH MATRIX SLICES ===
Dataset Split Sizes -> Train: 16200, Val: 4400, Test: 4312

=== STARTING TRAINING PROCESS ===
Epoch Complete. Avg Batch Training Loss: 0.6813

=== 